In [1]:
from pathlib import Path

import numpy as np

from nicht_riemann_data.transforms import spacings
from nicht_riemann_data.diagnostics import describe

print("OK")

OK


In [2]:
PROJECT_ROOT = Path.cwd()

RAW_FILE = PROJECT_ROOT / "data" / "raw" / "zeros1"

print("project:", PROJECT_ROOT)
print("raw file:", RAW_FILE)
print("exists:", RAW_FILE.exists())

assert RAW_FILE.exists()

project: /content/nicht-riemann-data
raw file: /content/nicht-riemann-data/data/raw/zeros1
exists: True


In [3]:
gamma = np.loadtxt(RAW_FILE, dtype=np.float64)

assert gamma.ndim == 1
assert len(gamma) == 100_000
assert np.all(np.isfinite(gamma))
assert np.all(np.diff(gamma) > 0)

delta = spacings(gamma)

assert delta.shape == (len(gamma) - 1,)
assert np.all(np.isfinite(delta))
assert np.all(delta > 0)

print("zeros    :", len(gamma))
print("spacings :", len(delta))
print("gamma    :", gamma[:5])
print("delta    :", delta[:5])

zeros    : 100000
spacings : 99999
gamma    : [14.13472514 21.02203964 25.01085758 30.42487613 32.93506159]
delta    : [6.8873145  3.98881794 5.41401855 2.51018546 4.65111657]


In [4]:
print("spacing statistics")
print("------------------")
print("n    :", len(delta))
print("min  :", np.min(delta))
print("max  :", np.max(delta))
print("mean :", np.mean(delta))
print("std  :", np.std(delta))

assert len(delta) == len(gamma) - 1
assert np.all(np.isfinite(delta))
assert np.all(delta > 0)

print()
print("PASS: valid positive spacing sequence")

spacing statistics
------------------
n    : 99999
min  : 0.014701476000482216
max  : 6.887314496999998
mean : 0.7490744184827048
std  : 0.32154023373868434

PASS: valid positive spacing sequence


In [5]:
SEED = 42
rng = np.random.default_rng(SEED)

mean_spacing = np.mean(delta)

surrogate = rng.exponential(
    scale=mean_spacing,
    size=len(delta),
)

assert surrogate.shape == delta.shape
assert np.all(np.isfinite(surrogate))
assert np.all(surrogate > 0)

print("surrogate n    :", len(surrogate))
print("target mean    :", mean_spacing)
print("surrogate mean :", np.mean(surrogate))
print("surrogate std  :", np.std(surrogate))

surrogate n    : 99999
target mean    : 0.7490744184827048
surrogate mean : 0.7516956806009978
surrogate std  : 0.7530397347617324


In [6]:
percentiles = [0, 1, 5, 25, 50, 75, 95, 99, 100]

original_q = np.percentile(delta, percentiles)
surrogate_q = np.percentile(surrogate, percentiles)

print("percentile comparison")
print("---------------------")

for p, original, surrogate_value in zip(
    percentiles,
    original_q,
    surrogate_q,
):
    print(
        f"{p:>3}% : "
        f"original={original:.6f}  "
        f"surrogate={surrogate_value:.6f}"
    )

percentile comparison
---------------------
  0% : original=0.014701  surrogate=0.000022
  1% : original=0.171941  surrogate=0.007487
  5% : original=0.290969  surrogate=0.038354
 25% : original=0.522533  surrogate=0.216963
 50% : original=0.714470  surrogate=0.520384
 75% : original=0.935618  surrogate=1.041341
 95% : original=1.318237  surrogate=2.251914
 99% : original=1.656738  surrogate=3.462187
100% : original=6.887314  surrogate=9.111655


In [7]:
BLOCK_SIZE = 1000

def block_statistics(values, block_size):
    n_blocks = len(values) // block_size

    means = []
    stds = []

    for i in range(n_blocks):
        block = values[
            i * block_size:(i + 1) * block_size
        ]

        means.append(np.mean(block))
        stds.append(np.std(block))

    return np.asarray(means), np.asarray(stds)


original_means, original_stds = block_statistics(
    delta,
    BLOCK_SIZE,
)

surrogate_means, surrogate_stds = block_statistics(
    surrogate,
    BLOCK_SIZE,
)

print("blocks:", len(original_means))

print()
print("block mean:")
print("  original min/max:",
      original_means.min(),
      original_means.max())
print("  surrogate min/max:",
      surrogate_means.min(),
      surrogate_means.max())

print()
print("block std:")
print("  original min/max:",
      original_stds.min(),
      original_stds.max())
print("  surrogate min/max:",
      surrogate_stds.min(),
      surrogate_stds.max())

blocks: 99

block mean:
  original min/max: 0.6705617749410012 1.406281801182
  surrogate min/max: 0.6880603702183947 0.8041705207918817

block std:
  original min/max: 0.2701797255264204 0.65639121478494
  surrogate min/max: 0.6772350073400477 0.8492774702909162


In [8]:
def autocorrelation(values, lag):
    x = values[:-lag]
    y = values[lag:]

    x = x - np.mean(x)
    y = y - np.mean(y)

    denominator = np.sqrt(
        np.sum(x * x) *
        np.sum(y * y)
    )

    return np.sum(x * y) / denominator


LAGS = [1, 2, 3, 4, 5, 10, 20, 50]

print("autocorrelation")
print("----------------")

for lag in LAGS:
    original_ac = autocorrelation(delta, lag)
    surrogate_ac = autocorrelation(surrogate, lag)

    print(
        f"lag={lag:>2} : "
        f"original={original_ac:+.8f}  "
        f"surrogate={surrogate_ac:+.8f}"
    )

autocorrelation
----------------
lag= 1 : original=-0.20445373  surrogate=+0.00233641
lag= 2 : original=+0.04328864  surrogate=+0.00399326
lag= 3 : original=+0.07563653  surrogate=-0.00068329
lag= 4 : original=+0.08815947  surrogate=+0.00392825
lag= 5 : original=+0.09569924  surrogate=+0.00218598
lag=10 : original=+0.12746541  surrogate=-0.00428843
lag=20 : original=+0.04803596  surrogate=-0.00292871
lag=50 : original=+0.12599568  surrogate=-0.00156585


In [9]:
comparison = {
    "mean": (
        np.mean(delta),
        np.mean(surrogate),
    ),
    "std": (
        np.std(delta),
        np.std(surrogate),
    ),
    "min": (
        np.min(delta),
        np.min(surrogate),
    ),
    "max": (
        np.max(delta),
        np.max(surrogate),
    ),
    "median": (
        np.median(delta),
        np.median(surrogate),
    ),
}

print("metric comparison")
print("-----------------")
print(f"{'metric':<10} {'original':>14} {'surrogate':>14}")

for name, (original, surrogate_value) in comparison.items():
    print(
        f"{name:<10} "
        f"{original:>14.8f} "
        f"{surrogate_value:>14.8f}"
    )

metric comparison
-----------------
metric           original      surrogate
mean           0.74907442     0.75169568
std            0.32154023     0.75303973
min            0.01470148     0.00002240
max            6.88731450     9.11165512
median         0.71447040     0.52038385


In [10]:
assert len(delta) == len(surrogate)

assert np.all(np.isfinite(delta))
assert np.all(np.isfinite(surrogate))

assert np.all(delta > 0)
assert np.all(surrogate > 0)

assert original_means.shape == surrogate_means.shape
assert original_stds.shape == surrogate_stds.shape

# Original and surrogate use the same mean-spacing scale.
mean_delta = np.mean(delta)
mean_surrogate = np.mean(surrogate)

assert np.isclose(
    mean_surrogate,
    mean_delta,
    rtol=0.02,
)

print("05_compare: PASS")
print()
print("Original and surrogate datasets are")
print("shape-compatible, finite, positive,")
print("and represented on the same mean-spacing scale.")

print()
print("mean original  :", mean_delta)
print("mean surrogate :", mean_surrogate)

05_compare: PASS

Original and surrogate datasets are
shape-compatible, finite, positive,
and represented on the same mean-spacing scale.

mean original  : 0.7490744184827048
mean surrogate : 0.7516956806009978
